# 03. FMA REAL Audio Validation

최종 매핑된 **FMA REAL 296곡**이 정상적으로 다운로드되었는지 검증한다.

검증 항목:

1. `fma_real_mapping.csv`의 296개 track과 실제 MP3 파일 개수 일치 여부
2. 예상 `track_id`와 실제 다운로드된 파일의 `track_id` 일치 여부
3. 0 byte / 비정상적으로 작은 MP3 파일 존재 여부
4. 각 MP3 파일의 디코딩 가능 여부
5. 실제 오디오 duration 확인
6. 최종 QC 요약표 생성

> 이 노트북은 REAL 데이터 검증용이다. 검증이 완료되면 다음 단계에서 Clean TTA FAKE와 결합하여 `master_manifest.csv`를 생성한다.


In [ ]:
from pathlib import Path
import subprocess
import shutil
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(
    "/Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project"
)

MAPPING_PATH = PROJECT_ROOT / "data/metadata/fma_real_mapping.csv"
AUDIO_DIR = PROJECT_ROOT / "data/raw/FMA/selected_30s"
REPORT_PATH = PROJECT_ROOT / "data/metadata/fma_real_audio_validation.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("MAPPING_PATH:", MAPPING_PATH)
print("AUDIO_DIR   :", AUDIO_DIR)


## 1. 매핑 파일과 다운로드된 MP3 개수 확인

`fma_real_mapping.csv`의 행 수와 실제 저장된 MP3 파일 수가 일치하는지 확인한다.


In [ ]:
mapping = pd.read_csv(MAPPING_PATH)
audio_files = sorted(AUDIO_DIR.rglob("*.mp3"))

print("===== FMA REAL AUDIO CHECK =====")
print("Mapping rows      :", len(mapping))
print("Unique track_id   :", mapping["track_id"].nunique())
print("Downloaded MP3    :", len(audio_files))


## 2. Track ID 일치 여부 확인

매핑 파일에서 기대되는 `track_id`와 실제 파일명의 `track_id`를 비교한다.

- `Missing = 0`
- `Extra = 0`

이면 정상이다.


In [ ]:
expected_ids = set(mapping["track_id"].astype(int).tolist())
downloaded_ids = {int(path.stem) for path in audio_files}

missing_ids = sorted(expected_ids - downloaded_ids)
extra_ids = sorted(downloaded_ids - expected_ids)

print("===== TRACK ID MATCH CHECK =====")
print("Expected :", len(expected_ids))
print("Found    :", len(downloaded_ids))
print("Missing  :", len(missing_ids))
print("Extra    :", len(extra_ids))

if missing_ids:
    print("\nMissing track IDs:")
    print(missing_ids)

if extra_ids:
    print("\nExtra track IDs:")
    print(extra_ids)


## 3. 파일 크기 기반 기본 무결성 검사

0 byte 파일이나 지나치게 작은 파일이 있는지 확인한다.

30초 MP3는 보통 수백 KB 이상이므로 여기서는 **100 KB 미만** 파일을 추가 점검 대상으로 표시한다.


In [ ]:
file_check = []

for path in audio_files:
    file_check.append({
        "track_id": int(path.stem),
        "path": str(path.relative_to(PROJECT_ROOT)),
        "size_bytes": path.stat().st_size,
    })

file_check = pd.DataFrame(file_check)

zero_byte_count = int((file_check["size_bytes"] == 0).sum())
small_files = file_check[file_check["size_bytes"] < 100_000].copy()

print("0 byte files   :", zero_byte_count)
print("100KB 미만 파일:", len(small_files))

display(file_check["size_bytes"].describe())

if len(small_files):
    display(small_files)


## 4. MP3 디코딩 및 실제 Duration 검사

각 MP3가 실제로 열리는지 확인하고 duration을 측정한다.

우선 `ffprobe`가 설치되어 있으면 이를 사용한다. 설치되어 있지 않으면 `librosa`를 사용하도록 시도한다.

FMA Large는 일반적으로 30초 clip으로 구성되므로 대부분 약 30초가 예상된다. 다만 원곡 자체가 30초보다 짧은 경우 더 짧을 수 있다.


In [ ]:
FFPROBE_AVAILABLE = shutil.which("ffprobe") is not None

print("ffprobe available:", FFPROBE_AVAILABLE)

if not FFPROBE_AVAILABLE:
    try:
        import librosa
        print("librosa available: True")
    except ImportError:
        print("librosa available: False")
        raise RuntimeError(
            "ffprobe 또는 librosa 중 하나가 필요합니다. "
            "Mac에서는 `brew install ffmpeg` 또는 현재 conda 환경에 librosa를 설치하세요."
        )


In [ ]:
def probe_audio_duration(path: Path):
    """Return (ok, duration_sec, error_message)."""
    if FFPROBE_AVAILABLE:
        cmd = [
            "ffprobe",
            "-v", "error",
            "-show_entries", "format=duration",
            "-of", "default=noprint_wrappers=1:nokey=1",
            str(path),
        ]
        result = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
        )

        if result.returncode != 0:
            return False, np.nan, result.stderr.strip()

        try:
            duration = float(result.stdout.strip())
            return True, duration, ""
        except ValueError:
            return False, np.nan, "duration 값을 float로 변환하지 못함"

    else:
        try:
            duration = float(librosa.get_duration(path=str(path)))
            return True, duration, ""
        except Exception as e:
            return False, np.nan, f"{type(e).__name__}: {e}"


audio_validation = []

for i, path in enumerate(audio_files, start=1):
    ok, duration_sec, error = probe_audio_duration(path)

    audio_validation.append({
        "track_id": int(path.stem),
        "path": str(path.relative_to(PROJECT_ROOT)),
        "size_bytes": path.stat().st_size,
        "decode_ok": ok,
        "duration_sec": duration_sec,
        "decode_error": error,
    })

    if i % 50 == 0 or i == len(audio_files):
        print(f"checked: {i}/{len(audio_files)}")

audio_validation = pd.DataFrame(audio_validation)

print("\n===== DECODE CHECK =====")
print("Decode success:", int(audio_validation["decode_ok"].sum()))
print("Decode failed :", int((~audio_validation["decode_ok"]).sum()))


In [ ]:
decode_failures = audio_validation[~audio_validation["decode_ok"]].copy()

if len(decode_failures):
    print("===== DECODE FAILURES =====")
    display(decode_failures)
else:
    print("모든 MP3 파일이 정상적으로 디코딩되었습니다.")

print("\n===== DURATION SUMMARY =====")
display(audio_validation["duration_sec"].describe())


## 5. Duration 이상값 확인

일반적인 FMA Large clip은 약 30초이다.

다만 FMA 생성 과정에서 원곡 자체가 30초 이하인 경우 전체 길이가 그대로 유지될 수 있으므로, 30초 미만 파일이 존재한다고 해서 바로 오류라고 판단하지 않는다.

여기서는 다음을 별도로 확인한다.

- 0초 이하
- 31초 초과
- 10초 미만


In [ ]:
invalid_duration = audio_validation[
    (audio_validation["decode_ok"]) &
    (
        (audio_validation["duration_sec"] <= 0) |
        (audio_validation["duration_sec"] > 31)
    )
].copy()

very_short = audio_validation[
    (audio_validation["decode_ok"]) &
    (audio_validation["duration_sec"] < 10)
].copy()

print("0초 이하 또는 31초 초과:", len(invalid_duration))
print("10초 미만             :", len(very_short))

if len(invalid_duration):
    display(invalid_duration)

if len(very_short):
    print("\n===== 10초 미만 파일 =====")
    display(very_short)


## 6. 매핑 정보와 오디오 QC 결과 결합

최종적으로 `fma_real_mapping.csv`와 실제 오디오 검증 결과를 `track_id` 기준으로 결합한다.


In [ ]:
real_validation = mapping.merge(
    audio_validation,
    on="track_id",
    how="left",
    validate="one_to_one",
)

real_validation["file_exists"] = real_validation["path"].notna()
real_validation["size_ok"] = real_validation["size_bytes"].fillna(0) >= 100_000

display(real_validation.head())


## 7. 최종 QC 요약

모든 핵심 항목이 정상인지 최종 확인한다.


In [ ]:
qc_summary = pd.DataFrame({
    "check": [
        "mapping_rows",
        "unique_track_id",
        "downloaded_mp3",
        "missing_track_id",
        "extra_track_id",
        "zero_byte_files",
        "under_100kb_files",
        "decode_success",
        "decode_failed",
        "duration_outside_0_to_31",
        "under_10sec_files",
    ],
    "value": [
        len(mapping),
        mapping["track_id"].nunique(),
        len(audio_files),
        len(missing_ids),
        len(extra_ids),
        zero_byte_count,
        len(small_files),
        int(audio_validation["decode_ok"].sum()),
        int((~audio_validation["decode_ok"]).sum()),
        len(invalid_duration),
        len(very_short),
    ]
})

display(qc_summary)

all_core_checks_pass = (
    len(mapping) == 296
    and mapping["track_id"].nunique() == 296
    and len(audio_files) == 296
    and len(missing_ids) == 0
    and len(extra_ids) == 0
    and zero_byte_count == 0
    and int((~audio_validation["decode_ok"]).sum()) == 0
)

print("===== FINAL RESULT =====")
print("Core QC PASS:", all_core_checks_pass)


## 8. 검증 결과 저장

곡별 QC 결과를 CSV로 저장한다. 이후 `master_manifest.csv` 생성 시 REAL 데이터의 검증 정보로 사용할 수 있다.


In [ ]:
real_validation.to_csv(
    REPORT_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("Saved:", REPORT_PATH)
print("Rows :", len(real_validation))


## 다음 단계

REAL 296곡의 QC가 통과하면 다음 단계는 다음과 같다.

```text
FMA REAL 296
      +
Clean Echoes TTA FAKE 3,162
      ↓
master_manifest.csv
      ↓
original_audio 단위 Train / Validation / Test split
```
